# Figure 9.20: Regression Fits Across Model Complexity

Low degree underfits, moderate degree tracks the trend, and high degree begins to chase noise.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def linspace(a, b, n):
    return np.linspace(a, b, int(n))


def base_curve(x):
    return 1.0 + 0.7 * x - 0.35 * x**2 + 0.06 * x**3


def poly_design(x, degree):
    return np.vstack([x**k for k in range(degree + 1)]).T


def ridge_fit(x, y, degree, lam):
    X = poly_design(x, degree)
    I = np.eye(degree + 1)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def poly_predict(theta, x):
    X = poly_design(np.asarray(x), len(theta) - 1)
    return X @ theta


def line_fit(x, y, lam=0.0):
    X = np.column_stack([np.ones_like(x), x])
    I = np.eye(2)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def line_predict(theta, x):
    x = np.asarray(x)
    return theta[0] + theta[1] * x


def poly_data(seed=0, n=40, obs_noise=0.3, gt_noise=0.0):
    rng = rng_from_seed(seed)
    x = np.linspace(-3, 3, n)
    y_true = base_curve(x)
    y_star = y_true + gt_noise * rng.normal(size=n)
    y = y_star + obs_noise * rng.normal(size=n)
    return x, y, y_true, y_star

def draw(noise=0.25, n=45, mid_degree=4, high_degree=12):
    x, y, _, _ = poly_data(seed=23, n=n, obs_noise=noise, gt_noise=0.0)
    xg = np.linspace(-3, 3, 300)
    low_fit = ridge_fit(x, y, 1, 0.0)
    mid_fit = ridge_fit(x, y, mid_degree, 0.0)
    high_fit = ridge_fit(x, y, high_degree, 0.0)
    fig, ax = plt.subplots(1, 1, figsize=(10.5, 5.5))
    ax.scatter(x, y, color="#334155", s=18, label="samples")
    ax.plot(xg, base_curve(xg), "--", color="#64748b", lw=2, label="true trend")
    ax.plot(xg, poly_predict(low_fit, xg), color="#2563eb", lw=2.5, label="degree 1")
    ax.plot(xg, poly_predict(mid_fit, xg), color="#059669", lw=2.5, label=f"degree {mid_degree}")
    ax.plot(xg, poly_predict(high_fit, xg), color="#dc2626", lw=2.5, label=f"degree {high_degree}")
    ax.set_title("All model complexities in one figure")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, ncol=3, loc="upper left")
    plt.show()

widgets.interact(
    draw,
    noise=widgets.FloatSlider(min=0.05, max=0.8, step=0.05, value=0.25, continuous_update=False),
    n=widgets.IntSlider(min=25, max=90, step=5, value=45, continuous_update=False),
    mid_degree=widgets.IntSlider(min=1, max=8, step=1, value=4, continuous_update=False),
    high_degree=widgets.IntSlider(min=1, max=14, step=1, value=12, continuous_update=False),
)
